# TNR v6 trên Google Colab
Notebook này chạy **lượt chính thức v6** (tiền đăng ký: `prereg/PREREG_v6.md`, đã khóa trước khi chạy). Kết quả và checkpoint được lưu trên **Google Drive**, nên nếu phiên Colab bị ngắt chỉ cần chạy lại các ô từ đầu: script bỏ qua những lần chạy đã xong và chạy tiếp.

**Cách chia việc:** mỗi thế giới (seed 90–95) độc lập, mất khoảng 1–2 giờ (3 khởi tạo × 14 nhánh). Mở 3 notebook/tài khoản song song, mỗi phiên một cặp thế giới (`90 91`, `92 93`, `94 95`), cùng ghi vào một thư mục Drive (hoặc mỗi phiên một thư mục rồi gộp).
**Runtime:** CPU chạy được; GPU không bắt buộc (mô hình rất nhỏ).
**Không** dán token GitHub vào notebook; ô cuối dùng `getpass` để nhập lúc chạy.

## 1. Kết nối Google Drive (lưu kết quả bền vững)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/tnr_v6'   # có thể đổi tên thư mục; mọi kết quả + checkpoint nằm ở đây
os.makedirs(OUT, exist_ok=True)
print('Kết quả sẽ lưu tại', OUT)

## 2. Lấy mã từ nhánh `from-claude` và cài thư viện

In [ ]:
!rm -rf /content/tnr
!git clone -b from-claude https://github.com/minhtri22/tnr.git /content/tnr
%cd /content/tnr
!pip -q install -r requirements.txt
import jax; print('JAX', jax.__version__, jax.devices())

## 3. Chạy (chọn thế giới cho phiên này)
Đổi `WORLDS` cho từng phiên. Có thể chạy lại ô này bao nhiêu lần tùy ý sau khi bị ngắt: tiến độ không mất.

In [ ]:
WORLDS = '90 91'      # phiên 1: '90 91'; phiên 2: '92 93'; phiên 3: '94 95'
!python src/run_v6.py --worlds $WORLDS --out "$OUT" 

## 4. Theo dõi tiến độ (tùy chọn)

In [ ]:
import glob
fs = sorted(glob.glob(OUT + '/world*_r*_*.json'))
print(len(fs), 'lần chạy đã xong (mỗi thế giới cần 3 × 14 = 42)')

## 5. Phân tích (khi cả 6 thế giới đã xong; nếu dùng nhiều thư mục Drive, liệt kê tất cả sau `--dirs`)

In [ ]:
!python src/analyze_v6.py --dirs "$OUT" 

## 6. (Tùy chọn) Đẩy kết quả về nhánh `from-claude`
Cần Personal Access Token **fine-grained**, chỉ cấp cho repo `minhtri22/tnr`, quyền *Contents: Read and write*. Token được nhập bằng `getpass`, không lưu vào mã.

In [ ]:
import getpass, base64, subprocess, shutil, os
token = getpass.getpass('GitHub token (không hiển thị): ')
hdr = 'AUTHORIZATION: basic ' + base64.b64encode(('x-access-token:' + token).encode()).decode()
def git(*a):
    return subprocess.run(['git', '-c', f'http.https://github.com/.extraheader={hdr}', *a], capture_output=True, text=True)
os.chdir('/content/tnr')
dst = 'runs/run_006_main'; os.makedirs(dst, exist_ok=True)
for f in os.listdir(OUT):
    p = os.path.join(OUT, f)
    if os.path.isfile(p): shutil.copy(p, dst)
    elif os.path.isdir(p): shutil.copytree(p, os.path.join(dst, f), dirs_exist_ok=True)
git('config', 'user.name', 'colab-run'); git('config', 'user.email', 'colab-run@users.noreply.github.com')
print(git('add', 'runs/run_006_main', 'results/results_v6.json', 'figures/v6_ket_qua.png').stderr)
print(git('commit', '-m', 'v6: kết quả chạy trên Colab').stdout)
print(git('push', 'origin', 'from-claude').stderr)
del token, hdr